# Playstyle Clusters — Manual Archetype Labeling

Cluster players by their lifetime playstyle vector — the exact vector
`PlayerSimilarity` uses for similarity — so the dashboard can label player
archetypes. This notebook is **exploration only**: nothing here feeds training
data or model features.

Workflow: run top to bottom, review the cluster profiles (centroid deviations
and nearest-centroid players), revise the provisional `cluster_labels` in the
parameters cell, then re-run the write cell to refresh the two artifacts the
runtime consumes:

- `data/processed/cluster_assignments.parquet` — `player_id`, `cluster_id`
- `data/processed/cluster_descriptions.json` — `{cluster_id: human label}`

On the next index build, `PlayerSimilarity.build` appends cluster membership
as a one-hot similarity feature and bakes the labels into `player_metadata.json`
for the directory/profile UI.

In [ ]:
from src.utils import load_env

load_env()

# ── Clustering parameters ─────────────────────────────────
# The cluster count is the explicit selection: the elbow/silhouette sweep
# below only informs it, never overrides it.
n_clusters = 5
random_state = 42
k_values = range(2, 13)

# ── PROVISIONAL manual labels (placeholders) ─────────────
# No auto-generated archetype names. Keys MUST match the fitted cluster ids
# (validated before writing). Review the UMAP plot and the cluster profiles
# below, then revise these labels and re-run the write cell to refresh the
# artifacts. The runtime surfaces these labels in the directory/profile UI.
cluster_labels = {
    "0": "PLACEHOLDER - describe cluster 0",
    "1": "PLACEHOLDER - describe cluster 1",
    "2": "PLACEHOLDER - describe cluster 2",
    "3": "PLACEHOLDER - describe cluster 3",
    "4": "PLACEHOLDER - describe cluster 4",
}

## Build the playstyle vectors

Query exactly the profile fields `PlayerSimilarity.build` queries, drop empty
player ids, then call `build_playstyle_matrix` — **without**
`cluster_assignments` so discovery is not skewed by a prior archetype. The
vector logic (one-hot handedness/backhand + `LIFETIME_PLAYSTYLE_COLS` + bio
embeddings, L2-normalized) lives in `src.models.similarity`; it is not
duplicated here.

In [ ]:
%matplotlib inline
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from kneed import KneeLocator
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from umap import UMAP

from src.constants import BRONZE_PROFILES_TABLE, DATA_PROCESSED
from src.db.client import to_dataframe
from src.models.similarity import LIFETIME_PLAYSTYLE_COLS, build_playstyle_matrix

profiles = to_dataframe(
    f"SELECT player_id, display_name, backhand, handedness, summary FROM {BRONZE_PROFILES_TABLE}"
)
profiles = profiles[profiles["player_id"] != ""].reset_index(drop=True)
print(f"{len(profiles)} profiled players")

playstyle_matrix = build_playstyle_matrix(profiles, query=to_dataframe)
features = playstyle_matrix.to_numpy(np.float32)
print(
    f"playstyle matrix: {playstyle_matrix.shape[0]} players x {playstyle_matrix.shape[1]} features"
)

In [ ]:
# ── K sweep: inertia (elbow) + silhouette ─────────────────
inertias: dict[int, float] = {}
silhouettes: dict[int, float] = {}
for k in k_values:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=random_state).fit(features)
    inertias[k] = float(kmeans.inertia_)
    silhouettes[k] = float(silhouette_score(features, kmeans.labels_))

In [ ]:
# ── Elbow + silhouette plots ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(inertias), list(inertias.values()), marker="o")
axes[0].set_title("Inertia by K (elbow)")
axes[0].set_xlabel("K")
axes[0].set_ylabel("Inertia")
axes[1].plot(list(silhouettes), list(silhouettes.values()), marker="o")
axes[1].set_title("Silhouette score by K")
axes[1].set_xlabel("K")
axes[1].set_ylabel("Silhouette score")
fig.tight_layout()
plt.show()

knee = KneeLocator(list(inertias), list(inertias.values()), curve="convex", direction="decreasing")
if knee.knee is not None:
    print(
        f"KneeLocator suggests K={knee.knee}; the selection stays the explicit "
        f"n_clusters={n_clusters} above."
    )
else:
    print("KneeLocator found no clear elbow; keep the explicit n_clusters selection.")

In [ ]:
# ── Fit the selected K and project into 2D ─────────────────
model = KMeans(n_clusters=n_clusters, n_init=10, random_state=random_state).fit(features)
assignments = pd.DataFrame({"player_id": profiles["player_id"], "cluster_id": model.labels_})

# Deterministic projection (fixed seed) purely for visual review.
umap_2d = np.asarray(
    UMAP(n_components=2, random_state=random_state).fit_transform(features),
    dtype=np.float32,
)

fig, ax = plt.subplots(figsize=(10, 7))
for cluster_id in range(n_clusters):
    mask = (assignments["cluster_id"] == cluster_id).to_numpy()
    label = cluster_labels.get(str(cluster_id), f"cluster {cluster_id}")
    ax.scatter(umap_2d[mask, 0], umap_2d[mask, 1], s=12, alpha=0.8, label=label)
ax.set_title(f"2D UMAP of playstyle vectors, K={n_clusters}")
ax.legend(loc="best", fontsize=8)
plt.show()

## Review cluster profiles

Two views to decide the archetype labels:

1. **Centroid deviations** — each cluster's centroid minus the mean centroid,
   restricted to `LIFETIME_PLAYSTYLE_COLS` (serve shape, aggression, return
   strength, clutch, surface preference). Positive = above-average for the
   archetype, negative = below.
2. **Nearest-centroid players** — the players closest to each centroid, the
   most representative faces of the cluster.

The labels are your call; the notebook never names archetypes.

In [ ]:
# ── Centroid deviations on LIFETIME_PLAYSTYLE_COLS ──────────
feature_columns = list(playstyle_matrix.columns)
lifetime_idx = [feature_columns.index(col) for col in LIFETIME_PLAYSTYLE_COLS]
centroids = model.cluster_centers_
deviation = pd.DataFrame(
    centroids[:, lifetime_idx] - centroids[:, lifetime_idx].mean(axis=0),
    columns=LIFETIME_PLAYSTYLE_COLS,
    index=[f"cluster {c} - {cluster_labels.get(str(c), '?')}" for c in range(n_clusters)],
)
print("Centroid deviations from the average playstyle (LIFETIME_PLAYSTYLE_COLS only):")
print(deviation.round(3).to_string())

# ── Nearest-centroid players per cluster ───────────────────
print("\nNearest-centroid players (most representative per cluster):")
samples_per_cluster = 5
for cluster_id in range(n_clusters):
    idx = np.where(assignments["cluster_id"] == cluster_id)[0]
    distance = np.linalg.norm(features[idx] - centroids[cluster_id], axis=1)
    nearest = idx[np.argsort(distance)[:samples_per_cluster]]
    print(f"\ncluster {cluster_id} - {cluster_labels.get(str(cluster_id), '?')}")
    for i in nearest:
        print(f"  {profiles['player_id'][i]:<14s} {profiles['display_name'][i]}")

## Write the runtime artifacts

The manual labels are validated against the fitted cluster ids — exact key
match — before anything is written, so the runtime never sees a label-less
cluster. `cluster_descriptions.json` uses string keys; `cluster_assignments`
carries exactly `player_id` + `cluster_id`.

In [ ]:
# ── Validate, then write the two runtime artifacts ─────────
fitted_ids = {str(c) for c in sorted(assignments["cluster_id"].unique())}
manual_ids = {str(k) for k in cluster_labels}
assert manual_ids == fitted_ids, (
    f"cluster_labels keys {sorted(manual_ids)} must exactly equal fitted "
    f"cluster ids {sorted(fitted_ids)}"
)

assignments_path = DATA_PROCESSED / "cluster_assignments.parquet"
descriptions_path = DATA_PROCESSED / "cluster_descriptions.json"
assignments.to_parquet(assignments_path, index=False)
descriptions_path.write_text(
    json.dumps({str(k): v for k, v in cluster_labels.items()}, indent=2, sort_keys=True)
)

print(f"Wrote {assignments_path} ({len(assignments)} rows, columns: {list(assignments.columns)})")
print(f"Wrote {descriptions_path}")

In [ ]:
# ── Next action ─────────────────────────────────────────────
print("After reviewing the labels above:")
print("  1. If any label needs changing, revise `cluster_labels` in the")
print("     parameters cell and re-run the write cell.")
print("  2. Rebuild the similarity index so the cluster one-hot feature and")
print("     labels join the FAISS index / metadata:")
print(
    '     uv run python -c "from src.models.similarity import PlayerSimilarity; '
    'PlayerSimilarity().build()"'
)
print("     (or run `just train`, which rebuilds the index via build_similarity_index).")